In [ ]:
---
tittle: "Microcompartment Database - Database structure - db.py file"
author: "Julia Monserrat Relano"
date: "2025-09-02"
format:
html:
    code-fold: true
---

// Cell 3 - Update table descriptions

# Database Structure - db.py file

SQL schema of the database:

![BMC Database Schema](PASTE IMAGE LATER)

[View ER Diagram](https://dbdiagram.io/d/DB_STRUCTURE_052025-682494635b2fc4582f9453e4)

## Tables Overview

**Core Protein Table**

* **Protein**: Stores protein sequence information and metadata
  - `prot_id` (int): Primary key, auto-generated
  - `prot_accession` (str): Unique accession (auto-generated as PR0001, PR0002, etc.)
  - `prot_seq` (str): Protein sequence (unique)
  - `is_canonical` (bool): True if canonical, False if isoform (default: True)
  - `struct_prot_type` (Enum): Type using StructProtType enum (HEXAMER, TRIMER, PENTAMER, OTHER)

**External Database Tables**

* **Xdatabase**: Describes external biological databases
  - `xref_db_id` (int): Primary key, auto-generated
  - `xref_db_name` (str): Database name (unique, required)
  - `xref_db_type` (Enum): Database type using DatabaseType enum (SEQUENCE, STRUCTURE, FUNCTION, TAXONOMY)
  - `xref_db_url` (str): Database URL (unique, required)

* **Xref**: Cross-reference records from external databases
  - `xref_id` (int): Primary key, auto-generated
  - `xref_acc_ext` (str): External database accession (unique, required)
  - `xref_db_id` (int): Foreign key to Xdatabase

**Linker Tables**

* **ProteinXref**: Links proteins to external database references (many-to-many bridge)
  - `prot_xref_id` (int): Primary key, auto-generated
  - `prot_id` (int): Foreign key to Protein
  - `xref_id` (int): Foreign key to Xref

* **ProteinName**: Links proteins to gene names (many-to-many bridge)
  - `prot_name_id` (int): Primary key, auto-generated
  - `prot_id` (int): Foreign key to Protein
  - `name_id` (int): Foreign key to Name

**Gene/CDS Tables**

* **Cds**: Coding DNA sequences linked to proteins
  - `cds_id` (int): Primary key, auto-generated
  - `cds_accession` (str): Unique CDS accession
  - `cds_seq` (str): DNA sequence (unique)
  - `prot_id` (int): Foreign key to Protein

* **Origin_cds**: Original sequences for engineered CDS
  - `origin_cds_id` (int): Primary key, auto-generated
  - `origin_cds_seq` (str): Original DNA sequence (unique)

* **CdsXref**: Links CDS to external database references (many-to-many bridge)
  - `cds_xref_id` (int): Primary key, auto-generated
  - `cds_id` (int): Foreign key to Cds
  - `xref_id` (int): Foreign key to Xref

**Modification Tables**

* **Modification**: Types of modifications applied to sequences
  - `modification_id` (int): Primary key, auto-generated
  - `modification_description` (str): Description of the modification
  - `modification_type` (Enum): Type using ModificationType enum (TRUNCATED, EXTENDED, FUSION, SYNTHETIC, MUTATED, DOMESTICATED)

* **CdsModification**: Links CDS to modifications (many-to-many bridge)
  - `cds_mod_id` (int): Primary key, auto-generated
  - `cds_id` (int): Foreign key to Cds
  - `modification_id` (int): Foreign key to Modification

**Protein Metadata Tables**

* **Name**: Gene/protein names
  - `name_id` (int): Primary key, auto-generated
  - `prot_name` (str): Protein name (required)

* **Isoforms**: Tracks canonical and isoform relationships
  - `iso_id` (int): Primary key, auto-generated
  - `prot_id_canonical` (int): Foreign key to canonical Protein
  - `prot_id_isoform` (int): Foreign key to isoform Protein

**Complex & Interaction Tables**

* **Complex**: BMC protein complexes
  - `complex_id` (int): Primary key, auto-generated
  - `complex_accession` (str): Unique complex accession (required)
  - `complex_type` (str): Complex classification (PDU, EUT, GRM, etc.)
  - `is_active` (bool): Whether complex is enzymatically active
  - `is_exp_tested` (bool): Whether assembly has been experimentally tested
  - `complex_source` (Enum): Source using ComplexSource enum (NATIVE, ENGINEERED, PREDICTED, THEORETICAL)

* **ProteinComplex**: Links proteins to complexes (many-to-many bridge)
  - `prot_id` (int): Foreign key to Protein
  - `complex_id` (int): Foreign key to Complex
  - `is_essential` (bool): Whether protein is essential for assembly (optional, default: False)
  - `stoichiometry` (int): Copy number in the complex (optional)

* **Interaction**: Types of protein-protein interactions
  - `interact_id` (int): Primary key, auto-generated
  - `interact_type` (str): Type of interaction

* **Ppi**: Protein-protein interactions between two specific proteins
  - `ppi_id` (int): Primary key, auto-generated (unique)
  - `interact_id` (int): Foreign key to Interaction
  - `prot_id_1` (int): Foreign key to first Protein
  - `prot_id_2` (int): Foreign key to second Protein

* **Ppi_complex**: Links protein-protein interactions to complexes (many-to-many bridge)
  - `ppi_complex_id` (int): Primary key, auto-generated
  - `ppi_id` (int): Foreign key to Ppi
  - `complex_id` (int): Foreign key to Complex

## Functions for Addition of Data to the Database

These functions check that information added to the database is not duplicate and relationships are correctly added.

**Tested and Ready to Use:**

* **add_protein(session, protseq, struct, canonical)** - Add a new protein or retrieve existing by sequence
* **add_xdatabase(session, xname, xurl=None, xtype=None, require_password=True)** - Add external database with password protection for untrusted sources
* **add_xref(session, xdb, protein, xrefacc, cds=None)** - Link protein and CDS to external database references

<div class="alert alert-block alert-danger"> 
<b>Not tested yet

* name_addition(protname, protein)
* cds_modificiation
* isoforms
* Ppi, interaction
* Protein_complex
</div>

## Functions to Create Database and Obtain Session

* **create_db(dbpath, force=False)** - Create database schema at specified path
* **get_session(dbpath)** - Get active SQLAlchemy session for database operations

